In [33]:
import pandas as pd
import json

# Step 1: Transform `omdb_major_movies` data

## Loading TMDB Raw Data

The raw OMDB JSON files collected in NB01b are loaded and converted into pandas DataFrames for further processing.

In [34]:
 with open("../data/raw/omdb_major_movies.json", "r") as f:
        major_omdb= json.load(f)
    

In [35]:
major_df = pd.DataFrame(major_omdb)
major_df.head()

,imdbID,Title,Year,Genre,Metascore,imdbRating
0,tt7286456,Joker,2019,"Crime, Drama, Thriller",59,8.3
1,tt1392190,Mad Max: Fury Road,2015,"Action, Adventure, Sci-Fi",90,8.1
2,tt1386697,Suicide Squad,2016,"Action, Adventure, Fantasy",40,5.9
3,tt0369610,Jurassic World,2015,"Action, Adventure, Sci-Fi",59,6.9
4,tt0451279,Wonder Woman,2017,"Action, Adventure, Fantasy",76,7.3


## Adding Studio Labels

A `studio_type` column is added to identify the group each movie belongs to.

This label is important for the later analysis, where movies are compared based on whether they are produced by major studios or non-major studios.

In [36]:
major_df["studio_type"] ="major"
major_df.head()

,imdbID,Title,Year,Genre,Metascore,imdbRating,studio_type
0,tt7286456,Joker,2019,"Crime, Drama, Thriller",59,8.3,major
1,tt1392190,Mad Max: Fury Road,2015,"Action, Adventure, Sci-Fi",90,8.1,major
2,tt1386697,Suicide Squad,2016,"Action, Adventure, Fantasy",40,5.9,major
3,tt0369610,Jurassic World,2015,"Action, Adventure, Sci-Fi",59,6.9,major
4,tt0451279,Wonder Woman,2017,"Action, Adventure, Fantasy",76,7.3,major


# Step 2: Transform `omdb_indie_movies` data

In [37]:
with open("../data/raw/omdb_indie_movies.json", "r") as f:
    indie_omdb= json.load(f)

In [38]:
indie_df = pd.DataFrame(indie_omdb)
indie_df.head()

,imdbID,Title,Year,Genre,Metascore,imdbRating
0,tt1431045,Deadpool,2016,"Action, Comedy, Sci-Fi",65,8.0
1,tt4154756,Avengers: Infinity War,2018,"Action, Adventure, Sci-Fi",68,8.4
2,tt4154796,Avengers: Endgame,2019,"Action, Adventure, Sci-Fi",78,8.4
3,tt2395427,Avengers: Age of Ultron,2015,"Action, Adventure, Sci-Fi",66,7.3
4,tt3498820,Captain America: Civil War,2016,"Action, Sci-Fi",75,7.8


In [39]:
indie_df["studio_type"] ="indie"
indie_df.head()

,imdbID,Title,Year,Genre,Metascore,imdbRating,studio_type
0,tt1431045,Deadpool,2016,"Action, Comedy, Sci-Fi",65,8.0,indie
1,tt4154756,Avengers: Infinity War,2018,"Action, Adventure, Sci-Fi",68,8.4,indie
2,tt4154796,Avengers: Endgame,2019,"Action, Adventure, Sci-Fi",78,8.4,indie
3,tt2395427,Avengers: Age of Ultron,2015,"Action, Adventure, Sci-Fi",66,7.3,indie
4,tt3498820,Captain America: Civil War,2016,"Action, Sci-Fi",75,7.8,indie


In [40]:
omdb_df = pd.concat(
    [major_df,indie_df],
    ignore_index = True
)
omdb_df.head()

,imdbID,Title,Year,Genre,Metascore,imdbRating,studio_type
0,tt7286456,Joker,2019,"Crime, Drama, Thriller",59,8.3,major
1,tt1392190,Mad Max: Fury Road,2015,"Action, Adventure, Sci-Fi",90,8.1,major
2,tt1386697,Suicide Squad,2016,"Action, Adventure, Fantasy",40,5.9,major
3,tt0369610,Jurassic World,2015,"Action, Adventure, Sci-Fi",59,6.9,major
4,tt0451279,Wonder Woman,2017,"Action, Adventure, Fantasy",76,7.3,major


# Step 3: Save the processed data to csv

In [41]:
omdb_df.to_csv("../data/processed/omdb_movies.csv",index=False)

# Step 4: Merging TMDB and OMDb Datasets
The processed TMDB and OMDb datasets are merged to combine movie metadata, production information, audience ratings, and critic ratings into a single analysis dataset.

In [42]:
tmdb_df = pd.read_csv("../data/processed/tmdb_movies.csv")
omdb_df = pd.read_csv("../data/processed/omdb_movies.csv")

### Merge Key

The two datasets are merged using IMDb ID:

- TMDB: `imdb_id`
- OMDb: `imdbID`

Before merging, duplicated or unnecessary columns are removed from the TMDB dataset to avoid redundant information and column conflicts.

In [43]:
# Remove duplicated or unnecessary columns from TMDB before merging.
# Use IMDb ID as the common identifier between TMDB and OMDb datasets.
# Inner join keeps only movies that have matching records in both datasets.
merged_df = pd.merge(
    tmdb_df.drop(columns=["studio_type", "title", "release_date"]),
    omdb_df,
    left_on="imdb_id",
    right_on="imdbID",
    how="inner"
)

### delete the replicated imdb_id

In [44]:
merged_df = merged_df.drop(columns=["imdb_id"])
merged_df.columns

Index(['budget', 'revenue', 'vote_average', 'vote_count', 'imdbID', 'Title',
       'Year', 'Genre', 'Metascore', 'imdbRating', 'studio_type'],
      dtype='str')

# Step 5: Feature Engineering

After merging TMDB and OMDb datasets, additional variables are created for the analysis,converting the original API variables into comparable measures that directly address the research question.

### Creating Audience Score

Two audience-based ratings are available from different platforms:

- TMDB `vote_average`
- IMDb `imdbRating`

Since both ratings represent general audience evaluations, I combine them by taking their average:

`audience_score = (TMDB vote_average + IMDb rating) / 2`

Combining the two sources allows the analysis to make use of all available audience rating information while reducing the influence of platform-specific differences. The resulting variable provides a more balanced estimate of overall audience perception across different movie platforms.

In [45]:
merged_df["audience_score"] = merged_df[["vote_average", "imdbRating"]].mean(axis=1)
merged_df.head()

,budget,revenue,vote_average,vote_count,imdbID,Title,Year,Genre,Metascore,imdbRating,studio_type,audience_score
0,55000000,1078958629,8.122,27972,tt7286456,Joker,2019,"Crime, Drama, Thriller",59.0,8.3,major,8.2110
1,150000000,378858340,7.635,24294,tt1392190,Mad Max: Fury Road,2015,"Action, Adventure, Sci-Fi",90.0,8.1,major,7.8675
2,175000000,749200054,5.917,22234,tt1386697,Suicide Squad,2016,"Action, Adventure, Fantasy",40.0,5.9,major,5.9085
3,150000000,1671537444,6.703,21727,tt0369610,Jurassic World,2015,"Action, Adventure, Sci-Fi",59.0,6.9,major,6.8015
4,149000000,823970682,7.207,21027,tt0451279,Wonder Woman,2017,"Action, Adventure, Fantasy",76.0,7.3,major,7.2535


### Creating Critic Score

The OMDb `Metascore` is originally measured on a 100-point scale.

To make it comparable with audience ratings measured on a 10-point scale, it is converted as:

`critic_score = Metascore / 10`

In [46]:
merged_df["metascore_10"] = merged_df["Metascore"] / 10
merged_df.head()

,budget,revenue,vote_average,vote_count,imdbID,Title,Year,Genre,Metascore,imdbRating,studio_type,audience_score,metascore_10
0,55000000,1078958629,8.122,27972,tt7286456,Joker,2019,"Crime, Drama, Thriller",59.0,8.3,major,8.2110,5.9
1,150000000,378858340,7.635,24294,tt1392190,Mad Max: Fury Road,2015,"Action, Adventure, Sci-Fi",90.0,8.1,major,7.8675,9.0
2,175000000,749200054,5.917,22234,tt1386697,Suicide Squad,2016,"Action, Adventure, Fantasy",40.0,5.9,major,5.9085,4.0
3,150000000,1671537444,6.703,21727,tt0369610,Jurassic World,2015,"Action, Adventure, Sci-Fi",59.0,6.9,major,6.8015,5.9
4,149000000,823970682,7.207,21027,tt0451279,Wonder Woman,2017,"Action, Adventure, Fantasy",76.0,7.3,major,7.2535,7.6


### Creating Audience–Critic Gap

The main outcome variable is the difference between audience and critic evaluations.

The gap is calculated as:

`gap = audience_score - critic_score`

Interpretation:

- Positive gap: audiences rate the movie higher than critics
- Negative gap: critics rate the movie higher than audiences

In [47]:
merged_df["gap"] = merged_df["audience_score"] - merged_df["metascore_10"]
merged_df.head()
merged_df.tail()

,budget,revenue,vote_average,vote_count,imdbID,Title,Year,Genre,Metascore,imdbRating,studio_type,audience_score,metascore_10,gap
872,9800000,2341534,6.639,2014,tt3716530,Elle,2016,"Crime, Drama, Thriller",89.0,7.1,indie,6.8695,8.9,-2.0305
873,4000000,64978931,8.150,2007,tt8267604,Capernaum,2018,Drama,75.0,8.4,indie,8.2750,7.5,0.7750
874,14000000,3621046,7.255,2006,tt4005402,Colonia,2015,"Biography, Drama, History",33.0,7.0,indie,7.1275,3.3,3.8275
875,30000000,109573511,7.204,2004,tt2261287,Leap!,2016,"Animation, Adventure, Comedy",48.0,6.8,indie,7.0020,4.8,2.2020
876,13000000,9101546,7.405,2000,tt4540710,Miss Sloane,2016,Drama,64.0,7.5,indie,7.4525,6.4,1.0525


# Step 6: Saving merged dataframe to csv

In [48]:
merged_df.to_csv("../data/processed/movies_clean.csv", index=False)